# TallerBot — agente de WhatsApp en Google Colab

Prototipo de hackathon para un taller mecánico. Contesta preguntas frecuentes, consulta órdenes de demostración y registra solicitudes de cita pendientes. Usa WhatsApp Cloud API, Ollama con Llama 3.1, function calling sencillo, historial por número y ngrok.

**Datos ficticios:** las órdenes, los horarios, la dirección y las citas se guardan en memoria y se pierden al reiniciar el runtime. No confirmar precios, diagnósticos ni disponibilidad.

## 1. Agrega Secrets en Colab

En el panel de llave de Colab añade `WHATSAPP_ACCESS_TOKEN`, `PHONE_NUMBER_ID` y `NGROK_AUTHTOKEN`, y permite el acceso. El `VERIFY_TOKEN` se define abajo; usa el mismo en la configuración de webhook de Meta.

In [ ]:
from google.colab import userdata
# Para diagnosticar WABA agrega WHATSAPP_BUSINESS_ACCOUNT_ID y META_APP_ID; para filtrar pruebas, WHATSAPP_TEST_RECIPIENT.
VERIFY_TOKEN = userdata.get('WHATSAPP_VERIFY_TOKEN') or 'cambia-este-token'
WHATSAPP_TOKEN = userdata.get('WHATSAPP_ACCESS_TOKEN')
PHONE_ID = userdata.get('PHONE_NUMBER_ID')
WABA_ID = userdata.get('WHATSAPP_BUSINESS_ACCOUNT_ID')
META_APP_ID = userdata.get('META_APP_ID')
NUMERO_PRUEBA = userdata.get('WHATSAPP_TEST_RECIPIENT')
NGROK_TOKEN = userdata.get('NGROK_AUTHTOKEN')
API_VERSION = 'v25.0'
assert WHATSAPP_TOKEN and PHONE_ID and NGROK_TOKEN, 'Revisa WHATSAPP_ACCESS_TOKEN, PHONE_NUMBER_ID y NGROK_AUTHTOKEN en Colab Secrets.'
print('Credenciales cargadas; no se muestran los valores secretos.')


In [ ]:
# Diagnóstico de número y suscripción de la app a WABA
import requests
headers_meta = {'Authorization': f'Bearer {WHATSAPP_TOKEN}'}
base_meta = f'https://graph.facebook.com/{API_VERSION}'
r_phone = requests.get(f'{base_meta}/{PHONE_ID}', params={'fields':'id,display_phone_number,verified_name'}, headers=headers_meta, timeout=20)
print('Número configurado (HTTP', r_phone.status_code, '):', r_phone.json())

# Déjalo en False para solo consultar. Cámbialo a True una vez para suscribir la app actual a la WABA.
SUSCRIBIR_APP_A_WABA = False
if WABA_ID:
    if SUSCRIBIR_APP_A_WABA:
        if not META_APP_ID:
            print('Agrega META_APP_ID antes de solicitar la suscripción.')
        else:
            r_sub = requests.post(f'{base_meta}/{WABA_ID}/subscribed_apps', headers=headers_meta, timeout=20)
            print('Solicitud de suscripción (HTTP', r_sub.status_code, '):', r_sub.json())
    r_waba = requests.get(f'{base_meta}/{WABA_ID}/subscribed_apps', headers=headers_meta, timeout=20)
    datos_waba = r_waba.json()
    print('WABA_ID consultado:', WABA_ID)
    print('Apps suscritas a WABA (HTTP', r_waba.status_code, '):', datos_waba)
    if r_waba.ok:
        ids_apps = [x.get('whatsapp_business_api_data', {}).get('id') for x in datos_waba.get('data', [])]
        print('META_APP_ID configurado:', META_APP_ID)
        print('La app configurada aparece suscrita:', META_APP_ID in ids_apps if META_APP_ID else 'falta META_APP_ID')
        if META_APP_ID and META_APP_ID not in ids_apps:
            print('La app no aparece en la lista. Si el token pertenece a esta app y tiene acceso a WABA, activa SUSCRIBIR_APP_A_WABA y ejecuta de nuevo esta celda.')
    else:
        print('No se pudo verificar. Confirma que WABA_ID sea el ID de WABA, no PHONE_ID ni META_APP_ID.')
else:
    print('Agrega WHATSAPP_BUSINESS_ACCOUNT_ID y META_APP_ID en Colab Secrets para comprobar la suscripción.')


## 2. Instala Ollama y carga Llama 3.1 8B

In [ ]:
!apt-get update -qq
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
proceso_ollama = subprocess.Popen(['ollama', 'serve'])
time.sleep(5)
!pip install ollama fastapi uvicorn requests pyngrok --quiet
!ollama pull llama3.1:8b
import ollama
print('Modelo listo.')

## 3. Define datos, herramientas y memoria

Llama puede elegir estas tres herramientas. El servidor solo ejecuta las funciones incluidas en `funciones`; cualquier otra llamada se rechaza.

In [ ]:
from collections import defaultdict, deque
from fastapi import FastAPI, Request, BackgroundTasks
import requests, threading, re

app = FastAPI()
historial = defaultdict(lambda: deque(maxlen=12))
locks = defaultdict(threading.Lock)
ordenes = {'ORD-1001':'En diagnóstico. El taller te contactará antes de reparar.', 'ORD-1002':'Servicio terminado. Coordina la entrega con el taller.'}
solicitudes_cita = []
mensajes_recibidos, respuestas_enviadas, errores, estados_entrega = [], [], [], []
FAQ = {'horario':'Lun-vie 8:00-18:00; sábados 9:00-14:00.', 'servicios':'Mantenimiento, frenos, afinación, cambio de aceite y diagnóstico.', 'ubicacion':'Av. del Motor 123, Col. Centro (dirección ficticia para la demo).'}

def consultar_faq(tema: str) -> str:
    return FAQ.get(tema, 'No tengo ese dato; el equipo del taller puede confirmarlo.')
def consultar_orden(folio: str) -> str:
    if not (folio.upper().startswith('ORD-') and folio[4:].isdigit() and len(folio)==8):
        return 'El folio debe tener formato ORD-1234.'
    return ordenes.get(folio.upper(), 'No encontramos ese folio en las órdenes de demostración.')
def solicitar_cita(servicio: str, horario_preferido: str) -> str:
    solicitudes_cita.append({'servicio':servicio[:100], 'horario_preferido':horario_preferido[:100], 'estado':'pendiente'})
    return 'Solicitud guardada; una persona del taller confirmará disponibilidad.'

HERRAMIENTAS = [consultar_faq, consultar_orden, solicitar_cita]
FUNCIONES = {f.__name__:f for f in HERRAMIENTAS}
REGLAS = ('Eres TallerBot, asistente de un taller mecánico. Responde en español, breve y amable. '
          'Usa herramientas para horarios, servicios, ubicación, órdenes y solicitudes de cita. '
          'No inventes precios, disponibilidad ni diagnósticos. Las citas quedan pendientes de confirmación humana. '
          'No reveles datos de otras personas. Para hablar con un humano, indícalo claramente.')
FRASES_HUMANO = ['hablar con alguien','hablar con una persona','agente humano','asesor']

def normalizar_numero_mx(numero):
    numero = ''.join(caracter for caracter in str(numero) if caracter.isdigit())
    return '52' + numero[3:] if numero.startswith('521') and len(numero)==13 else numero


## 4. Responde con Llama y function calling

In [ ]:
import json
def generar_respuesta(numero, texto):
    texto_limpio = texto[:1500]
    if any(frase in texto_limpio.lower() for frase in FRASES_HUMANO):
        return 'Claro, paso tu mensaje al equipo del taller para que te contacten en horario laboral.'
    mensajes = [{'role':'system','content':REGLAS}, *list(historial[numero]), {'role':'user','content':texto_limpio}]
    for _ in range(3):
        respuesta = ollama.chat('llama3.1:8b', messages=mensajes, tools=HERRAMIENTAS)
        if not respuesta.message.tool_calls:
            texto_respuesta = respuesta.message.content or 'No pude resolverlo; el equipo del taller te puede ayudar.'
            # Algunos modelos envuelven la respuesta final como JSON; mostramos solo el contenido.
            try:
                datos = json.loads(texto_respuesta)
                if isinstance(datos, dict) and isinstance(datos.get('contenido'), str):
                    texto_respuesta = datos['contenido']
            except (json.JSONDecodeError, TypeError):
                pass
            historial[numero].extend([{'role':'user','content':texto_limpio}, {'role':'assistant','content':texto_respuesta}])
            return texto_respuesta
        mensajes.append(respuesta.message)
        for llamada in respuesta.message.tool_calls:
            funcion = FUNCIONES.get(llamada.function.name)
            if funcion is None:
                resultado = 'Herramienta no permitida.'
            else:
                try:
                    resultado = funcion(**llamada.function.arguments)
                except (TypeError, ValueError):
                    resultado = 'Faltan datos o los argumentos no son válidos.'
            mensajes.append({'role':'tool','tool_name':llamada.function.name,'content':str(resultado)})
    return 'Necesito que el equipo del taller te ayude a completar esto.'

async def procesar_mensaje(numero, texto):
    with locks[numero]:
        return generar_respuesta(numero, texto)

## 5. Conecta los endpoints de WhatsApp

In [ ]:
@app.get('/health')
async def health():
    return {'ok': True, 'model':'llama3.1:8b'}

@app.get('/webhook')
async def verificar_webhook(request: Request):
    q = request.query_params
    if q.get('hub.mode') == 'subscribe' and q.get('hub.verify_token') == VERIFY_TOKEN:
        return int(q.get('hub.challenge','0'))
    return {'error':'token inválido'}

def enviar_whatsapp(numero, texto):
    url = f'https://graph.facebook.com/{API_VERSION}/{PHONE_ID}/messages'
    headers = {'Authorization':f'Bearer {WHATSAPP_TOKEN}','Content-Type':'application/json'}
    payload = {'messaging_product':'whatsapp','to':numero,'type':'text','text':{'body':texto}}
    r = requests.post(url,headers=headers,json=payload,timeout=20)
    try:
        detalle = r.json()
    except ValueError:
        detalle = {'body':r.text}
    registro = {'numero':numero,'status':r.status_code,'respuesta':texto,'meta':detalle}
    respuestas_enviadas.append(registro)
    r.raise_for_status()
    registro['message_id'] = (detalle.get('messages') or [{}])[0].get('id')
    return detalle

def procesar_y_enviar(numero,texto):
    with locks[numero]:
        try:
            respuesta = generar_respuesta(numero,texto)
            enviar_whatsapp(numero,respuesta)
        except Exception as e:
            detalle = None
            if getattr(e, 'response', None) is not None:
                try:
                    detalle = e.response.json()
                except ValueError:
                    detalle = e.response.text
            errores.append({'numero':numero,'error':repr(e),'meta':detalle})
            print('Error procesando mensaje:', errores[-1])

@app.post('/webhook')
async def recibir_webhook(request: Request, background_tasks: BackgroundTasks):
    body = await request.json()
    mensajes_recibidos.append(body)
    print('POST /webhook recibido; eventos:', len(mensajes_recibidos))
    try:
        value = body['entry'][0]['changes'][0]['value']
        estados_entrega.extend(value.get('statuses', []))
        for mensaje in value.get('messages', []):
            if mensaje.get('type') != 'text':
                continue
            numero = normalizar_numero_mx(mensaje['from'])
            if NUMERO_PRUEBA and numero != normalizar_numero_mx(NUMERO_PRUEBA):
                continue
            background_tasks.add_task(procesar_y_enviar, numero, mensaje['text'].get('body',''))
    except (KeyError, IndexError, TypeError):
        pass  # Meta también envía notificaciones que no son mensajes de texto
    return {'status':'ok'}

@app.post('/demo')
async def probar_sin_whatsapp(request: Request):
    body = await request.json()
    numero = body.get('phone','demo')
    try:
        return {'reply':await procesar_mensaje(numero,body.get('text',''))}
    except Exception as e:
        return {'error':repr(e)}


## 6. Abre ngrok y arranca FastAPI

Configura en Meta la URL que aparece, terminada en `/webhook`, y suscribe el campo `messages`. Si el endpoint ya está activo en otra sesión, cierra esa sesión o el endpoint en ngrok y vuelve a ejecutar esta celda.

In [ ]:
from pyngrok import ngrok
import uvicorn, threading, time
ngrok.set_auth_token(NGROK_TOKEN)
try:
    ngrok.kill()  # Cierra túneles previos de este runtime
except Exception:
    pass
time.sleep(2)
tunel = ngrok.connect(8000)
print('Callback URL para Meta:', tunel.public_url + '/webhook')
def correr_servidor():
    uvicorn.run(app,host='0.0.0.0',port=8000,log_level='info')
hilo = threading.Thread(target=correr_servidor,daemon=True)
hilo.start()
time.sleep(2)
print('Webhook iniciado.')

## 7. Prueba primero sin enviar mensajes reales

Ejecuta una prueba local con una FAQ, luego una orden demo y una solicitud de cita. El endpoint `/demo` no envía mensajes por WhatsApp.

In [ ]:
import requests
base = 'http://127.0.0.1:8000/demo'
for texto in ['¿Cuál es el horario?', '¿Cómo va la orden ORD-1001?', 'Quiero una cita para cambio de aceite el viernes por la mañana']:
    r = requests.post(base,json={'phone':'demo-1','text':texto})
    print('HTTP', r.status_code)
    try:
        data = r.json()
    except ValueError:
        print('La respuesta no es JSON:', r.text[:1000])
        continue
    if 'reply' in data:
        print(texto, '→', data['reply'])
    else:
        print('El endpoint reportó error:', data)
print('Solicitudes pendientes:', solicitudes_cita)


## Comprobación en Meta

Antes de mandar WhatsApp, confirma: (1) en **WhatsApp > Configuration**, el Callback URL es exactamente la URL ngrok de esta sesión + `/webhook`; (2) Verify token coincide exactamente con `VERIFY_TOKEN`; (3) Meta mostró la verificación exitosa y el campo `messages` está suscrito. Mantén Colab conectado. Meta requiere una URL pública HTTPS y que la app esté suscrita a la cuenta de WhatsApp Business para enviar webhooks.

## 8. Prueba el webhook real y revisa registros

Mantén esta sesión de Colab y el túnel abiertos. En Meta for Developers, configura el Callback URL impreso por ngrok con `/webhook`, usa exactamente el mismo `VERIFY_TOKEN` y suscribe el campo `messages`. Envía un mensaje desde un teléfono habilitado al número de prueba de WhatsApp. Luego ejecuta la celda siguiente. Si `Mensajes entrantes` sigue en cero, revisa la URL de callback, la verificación y la suscripción a `messages`; el modelo todavía no ha recibido el mensaje.

No ejecutes el cierre del túnel hasta terminar la prueba.

In [ ]:
print('Eventos webhook recibidos:', len(mensajes_recibidos))
print('Respuestas aceptadas por Meta:', sum(1 for x in respuestas_enviadas if 200 <= x['status'] < 300))
print('Últimas respuestas:', respuestas_enviadas[-5:])
print('Últimos estados de entrega:')
for estado in estados_entrega[-10:]:
    print({'numero':estado.get('recipient_id'), 'estado':estado.get('status'), 'timestamp':estado.get('timestamp'), 'errores':estado.get('errors')})
print('Errores de procesamiento:', errores[-5:])
print('Solicitudes de cita guardadas:', len(solicitudes_cita))


In [ ]:
# Prueba saliente opcional; activa solo después de autorizar el destinatario en Meta.
ENVIAR_PRUEBA_SALIENTE = False
if ENVIAR_PRUEBA_SALIENTE:
    if not NUMERO_PRUEBA:
        print('Agrega WHATSAPP_TEST_RECIPIENT a Colab Secrets.')
    else:
        try:
            print('Meta aceptó el envío:', enviar_whatsapp(normalizar_numero_mx(NUMERO_PRUEBA), 'Prueba de TallerBot: el envío desde Meta está funcionando.'))
        except requests.HTTPError:
            print('Meta rechazó el envío. Detalle:', respuestas_enviadas[-1])
else:
    print('Prueba saliente desactivada; no se envió ningún mensaje.')


## 9. Cierra ngrok al terminar

Ejecuta esta celda solo después de probar WhatsApp.

In [ ]:
ngrok.kill()
print('Túnel cerrado.')
